# Dig into the SF calculations from Dan and X

In [1]:
# imports
from importlib import reload
import os

import xarray
import pandas

import numpy as np
# import fsspec
import matplotlib
import matplotlib.pyplot as plt
import gsw_xarray as gsw
from xhistogram.xarray import histogram

from profiler.loading.pymatreader import pymatreader

from strucFunct2_ai import timescale

from profiler import gliderdata
from profiler import profilerpairs
from cugn import io as cugn_io
from cugn import utils as cugn_utils
from cugn import plotting as cugn_plotting

import qg_utils
import strucFunct2_ai
import glider_io

# Load up

## Dan

In [2]:
idg_datafile = os.path.join(os.getenv('OS_SPRAY'), 'ARCTERX', 'Leg2', 'dr_2gliders.mat')

In [3]:
d = pymatreader.read_mat(idg_datafile)
d.keys()

dict_keys(['__header__', '__version__', '__globals__', 'lat', 'lon', 'missid', 'time', 'u', 'v', 'x', 'y'])

In [4]:
d['u'].shape

(100, 625)

## X

In [5]:
dataset = 'ARCTERX-2025'
profilers = glider_io.load_dataset(dataset)
profilers

Loading Sprays
calc_dist_offset: theta=1.5707963267948966 rad, 90.0 deg
calc_dist_offset: theta=1.5707963267948966 rad, 90.0 deg
Using lonendpts: (129.9167, 129.9167)
Using latendpts: (20.3332, 20.3334)
calc_dist_offset: theta=1.5707963267948966 rad, 90.0 deg
calc_dist_offset: theta=1.5707963267948966 rad, 90.0 deg


/home/xavier/Projects/Oceanography/python/profiler/profiler/profilerpairs.py:552: RuntimeWarning: Mean of empty slice
  avg_r.append(np.nanmean(self.r[in_r]))
/home/xavier/Projects/Oceanography/python/profiler/profiler/profilerpairs.py:553: RuntimeWarning: Mean of empty slice
  avg_S1.append(np.nanmean(self.S1[in_r]))
/home/xavier/miniconda3/envs/ocean/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/xavier/miniconda3/envs/ocean/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/xavier/Projects/Oceanography/python/profiler/profiler/profilerpairs.py:556: RuntimeWarning: Mean of empty slice
  avg_S2.append(np.nanmean(self.S2[in_r]))
/home/xavier/Projects/Oceanography/python/profiler/profiler/profilerpairs.py:558: RuntimeWarning: Mean of emp

[SprayData object for ARCTERX-Leg2
   Mission ID: 25203301
   Number of profiles: 310
   Time range: 2025-02-03 01:55:07.999998093 to 2025-04-10 23:52:25.249999762
   In field? True  ADCP on? True  Variables:
     depth: (100,)
     time: (310,)
     lat: (310,)
     lon: (310,)
     distE: (310,)
     distN: (310,)
     s: (310, 100)
     t: (310, 100)
     udop: (310, 100)
     vdop: (310, 100),
 SprayData object for ARCTERX-Leg2
   Mission ID: 25203801
   Number of profiles: 322
   Time range: 2025-02-03 01:50:59.999997854 to 2025-04-10 23:18:19.250000477
   In field? True  ADCP on? True  Variables:
     depth: (100,)
     time: (322,)
     lat: (322,)
     lon: (322,)
     distE: (322,)
     distN: (322,)
     s: (322, 100)
     t: (322, 100)
     udop: (322, 100)
     vdop: (322, 100)]

In [6]:
profilers[0].profile_id

array([ 24,  25,  26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,
        37,  38,  39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,
        50,  51,  52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,
        63,  64,  65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,
        76,  77,  78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,
        89,  90,  91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101,
       102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114,
       115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127,
       128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140,
       141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153,
       154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166,
       167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179,
       180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192,
       193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 20

# Construct pairs

In [7]:
max_time = 7.
gPairs = profilerpairs.ProfilerPairs(profilers, max_time=max_time, debug=False, randomize=True)

Using lonendpts: (np.float64(129.93323750000002), np.float64(129.93323750000002))
Using latendpts: (np.float64(20.365377499999997), np.float64(20.365577499999997))
calc_dist_offset: theta=1.5707963267948966 rad, 90.0 deg
calc_dist_offset: theta=1.5707963267948966 rad, 90.0 deg


In [8]:
gPairs

ProfilerPair object for the following datasets:
 ARCTERX-Leg2, SprayData 25203301 
 ARCTERX-Leg2, SprayData 25203801 
  Number of pairs: 945
  Time range: 2025-02-03 01:50:59.999997854 to 2025-04-10 23:52:25.249999762

## Print the IDs of the pairs

In [12]:
df = pandas.DataFrame()
df['missid_0'] = gPairs.data('missida', 0)
df['missid_1'] = gPairs.data('missida', 1)
#
df['profid_0'] = gPairs.data('profile_id', 0)
df['profid_1'] = gPairs.data('profile_id', 1)
# 
df.head()

,missid_0,missid_1,profid_0,profid_1
0,25203801,25203301,45,24
1,25203301,25203801,24,46
2,25203301,25203801,24,47
3,25203801,25203301,48,24
4,25203301,25203801,25,46


In [13]:
df.to_csv('pair_IDs.csv')